In [1]:
# ============================================================
# Wound Closure Movie using Fisher-KPP Model
# Google Colab version
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import matplotlib.animation as animation

# -----------------------------
# 1. Parameters
# -----------------------------
N = 160              # grid size (NxN)
D = 0.22             # diffusion / migration rate
r = 0.06             # proliferation rate
dx = 1.0             # spatial step
dt = 0.15            # time step
steps = 700          # total number of simulation steps
wound_width = 40     # initial wound width
save_every = 5       # save a frame every 5 steps

# -----------------------------
# 2. Initial condition
# -----------------------------
# u = cell density
# 1 = fully covered by cells
# 0 = empty wound area

u = np.ones((N, N))

center = N // 2
half_width = wound_width // 2

# Create a vertical wound gap
u[:, center-half_width:center+half_width] = 0.0

# Small random noise
u += 0.01 * np.random.rand(N, N)
u = np.clip(u, 0, 1)

# -----------------------------
# 3. Laplacian with no-flux boundaries
# -----------------------------
def laplacian(Z, dx=1.0):
    Zp = np.pad(Z, 1, mode='edge')  # no-flux (Neumann-like) padding
    return (
        Zp[2:, 1:-1] + Zp[:-2, 1:-1] +
        Zp[1:-1, 2:] + Zp[1:-1, :-2] -
        4 * Zp[1:-1, 1:-1]
    ) / dx**2

# -----------------------------
# 4. Run simulation and store frames
# -----------------------------
frames = []
wound_area = []

for step in range(steps):
    diffusion = D * laplacian(u, dx)
    growth = r * u * (1 - u)

    u = u + dt * (diffusion + growth)
    u = np.clip(u, 0, 1)

    # wound area = low-density region
    wound_area.append(np.sum(u < 0.5))

    if step % save_every == 0:
        frames.append(u.copy())

print(f"Saved {len(frames)} frames.")

# -----------------------------
# 5. Make animation
# -----------------------------
fig, ax = plt.subplots(figsize=(6, 6))
im = ax.imshow(frames[0], cmap='viridis', vmin=0, vmax=1)
title = ax.set_title("Wound Closure Simulation (t = 0)")
ax.axis("off")
cbar = plt.colorbar(im, ax=ax)
cbar.set_label("Cell density")

def update(frame_idx):
    im.set_data(frames[frame_idx])
    title.set_text(f"Wound Closure Simulation (frame = {frame_idx})")
    return [im, title]

anim = FuncAnimation(
    fig, update, frames=len(frames), interval=120, blit=False
)

plt.close(fig)

# Display movie inside Colab
HTML(anim.to_html5_video())

Saved 140 frames.
